# 노래 구간 비교 실험: 보컬 분리 전/후 인식 (이슈 #18)

곡의 한 구간을 잘라 **보컬만 분리**하고(Demucs), 원곡과 분리한 보컬을 **같은 조건으로 인식해 정답과 비교**합니다. 터미널 명령(`scripts/separate_clip.py`, `scripts/compare_clips.py`)과 같은 모듈(`separate.py`, `clip_compare.py`)을 쓰며, 이 노트북은 **듣기·표 확인·그림**을 한 화면에서 하려는 것입니다. 절차와 기록 양식은 `docs/노래_구간_비교_실험_절차.md`, 판단은 검토 문서 13.5입니다.

**순서**: ① 설정 → ② 구간 정하기(`CLIPS`) → ③ 분리 실행 → ④ 들어 보기 → ⑤ 정답 파일 준비(편집기로 직접 받아 적기) → ⑥ 비교 실행 → ⑦ 스펙트로그램.

**주의 (공개 저장소)**
- 곡·구간·정답·인식 문장은 모두 git 제외 폴더(`data/`, `results/`)에만 저장됩니다. 이 노트북의 **출력에는 오디오 플레이어(곡 소리가 base64로 저장됨)가 들어갈 수 있으므로** 커밋하기 전에 출력을 지우고 `scripts/scan_outputs.py`로 확인합니다(정리본만 커밋).
- 가사(정답, 인식 문장)는 이 노트북 화면에 찍지 않도록 만들어져 있습니다. 점수 숫자만 나옵니다.
- 분리와 인식은 모두 내 컴퓨터에서만 돌고 외부 API로 나가지 않습니다(과금 없음).
- `.py`를 고친 뒤에는 자동으로 다시 읽도록 `autoreload`를 켜 두었지만, 이상하면 커널을 재시작합니다.

## 이 노트북은 무엇을 하는가 (한눈에 보기, 2026-11-03 저녁)

**질문 하나**: 노래에서 반주를 떼어 낸 목소리(분리 보컬)가 원곡 그대로보다 Whisper(STT)에게 **가사를 더 잘 받아 적게 하는가?** 그렇다면 그 문장으로 연습하는 흐름(이슈 #31·#32)으로 이어 가고, 아니라면 분리 없이 가는 근거가 됩니다. 이 노트북은 그 질문을 **내 곡 구간 8개의 숫자로 확인**하는 곳입니다(이슈 #18).

### 전체 순서

| 단계 | 하는 일 | 걸리는 시간 |
|---|---|---|
| ② `CLIPS` | 곡 번호·시작·길이로 구간을 정함 | 직접 |
| ③a 자르기 | 구간을 잘라 `cNN_mix.wav` 저장 | 몇 초 |
| ④ 듣기 | 원곡 구간이 가사 부분인지 귀로 확인 | 구간당 1분 |
| ③b 분리 | Demucs로 보컬만 분리해 `cNN_vocals.wav` 저장 (별도 프로세스) | 구간당 1~2분 |
| ⑤ 정답 | 원곡을 듣고 가사를 받아 적음 (`truth/cNN.txt`, 화면에는 단어 수만) | 구간당 5~10분 |
| **⑥ 비교** | **원곡 vs 보컬을 같은 조건으로 인식해 정답과 비교** | 조건에 따라 5~40분 |
| ⑦ 그림 | 원곡과 보컬의 스펙트로그램 | 몇 초 |

### ⑥ 비교 실행은 정확히 무엇을 비교하는가

비교의 **핵심은 소리**입니다. 나머지는 그 결론이 조건에 따라 달라지는지 보려고 바꿔 보는 값입니다.

| 바꾸는 값 | 값 | 보려는 것 |
|---|---|---|
| **소리** | 원곡 구간(`mix`) vs 분리한 보컬(`vocals`) | **분리가 도움이 되는가** (핵심) |
| 모델 크기 | `tiny`, `base`, `small` | 큰 모델은 반주에 더 강해서 분리의 이득이 줄어드는가 [추정] |
| VAD | 끔 / 켬 | 말소리 없는 구간을 거르는 처리가 노래에서 도움이 되는가 |
| `temperature` (추가 예정) | 기본(폴백) / 0.0 고정 | 결과가 실행마다 흔들리는 정도 (아래 "지금까지 결과") |

- **정답**: 내가 **원곡을 들으며** 받아 적은 가사입니다(분리 결과에 정답이 끌려가지 않게).
- **점수**: 인식된 문장과 정답을 단어 단위로 비교한 오류율 **WER**(치환·삭제·삽입 ÷ 정답 단어 수)와 글자 단위 **CER**. 낮을수록 정답에 가깝습니다. 1을 넘을 수 있습니다. 전체는 **micro**(오류 합 ÷ 정답 단어 합)를 기본으로, 구간별 평균 **macro**를 보조로 봅니다.
- 한 구간 × 한 조건 = 인식 한 번 = 결과 한 줄입니다(`results/song_compare/metrics.csv`). 인식된 문장(가사)은 `texts.csv`에만 저장되고 화면에는 나오지 않습니다.
- **점수는 발음이나 노래 실력이 아니라, STT가 정답 가사를 얼마나 되살렸는가**입니다.

### 왜 하는가
1. **가설을 검증**하려고 합니다. "분리하면 좋아진다"는 직관이고, 분리가 만든 어색한 소리가 오히려 인식을 해칠 수도 있습니다(검토 문서 15.4).
2. **대조군**이 있습니다. c07(아카펠라)은 반주가 없어서 분리가 필요 없는 경우입니다. 여기서도 분리가 크게 달라지면 분리 자체의 영향을 의심합니다.
3. 결정이 걸려 있습니다. 분리가 쓸 만하면 #31·#32로 가고, 아니면 계획서의 축소 규칙대로 분리 없이 가거나 구간을 줄입니다.

### 지금까지 결과 (참고 baseline, tiny, VAD 끔, 8구간·정답 231단어)

| 실행 | 원곡 오류 / micro WER | 보컬 오류 / micro WER |
|---|---|---|
| 터미널(`compare_clips.py`) | 117 / 0.506 | 140 / 0.606 |
| 이 노트북 ⑥ | 126 / 0.545 | 160 / 0.693 |

- **같은 입력인데 두 실행의 값이 달랐습니다.** 원인을 확인했습니다 [확인]: faster-whisper가 기본으로 쓰는 **온도 폴백**(어려운 소리에서 무작위 샘플링으로 재시도)이 노래에서 발동합니다. 같은 16개 파일을 두 번 인식하면 **4개의 문장이 달랐고, `temperature=0.0`이면 16개 모두 같았습니다.** 연습 문장 30건에서는 이런 일이 없었습니다(말소리라서 폴백이 거의 안 걸림).
- 따라서 **실행 간 차이(0.04~0.09)가 원곡-보컬 차이(0.10~0.15)와 비슷한 크기**여서, 지금 값만으로 "분리하면 나빠진다"고 말할 수 없습니다. 결론은 보류입니다.

### 커밋 직전에 하던 일 (정리)
1. 자르기(`cut_clip`)와 분리(`separate_clip`)를 나눔: c05·c06이 가사 구간이 아니어서 분리 시간이 낭비된 것을 계기로, 먼저 듣고 맞는 구간만 분리하도록 바꿈.
2. ③b를 별도 프로세스로 바꿈: 분리(torch)와 인식(Whisper)을 한 커널에서 쓰면 OpenMP 충돌로 커널이 죽음(이슈노트 #15).
3. 위 결과 불일치의 원인(온도 폴백)을 찾아 문서를 정정함(검토 문서 13.5).
4. **C안 결정**: `temperature=0.0` 고정을 주 결과로 하고 기본 설정 3회 범위를 민감도 확인으로 병행합니다. 구현은 커밋 정리 뒤에 합니다.
5. 변경 사항을 주제별로 커밋 정리.

**다음 할 일**은 `docs/작업_일지.md` 3장의 11-03 항목에 있습니다.

In [ ]:
# ① 설정: 프로젝트 루트로 이동하고 모듈을 불러옵니다. (이 노트북은 experiments/ 안에 있어서 상대경로 data/...가 맞도록 루트로 옮깁니다.)
import os
import sys
from pathlib import Path

root = Path.cwd()
while not (root / "separate.py").exists() and root != root.parent:
    root = root.parent                      # 위 폴더로 올라가며 프로젝트 루트(separate.py가 있는 곳)를 찾는다
os.chdir(root)
sys.path.insert(0, str(root))

try:                                        # .py를 고치면 커널을 재시작하지 않아도 다시 읽게 한다(노트북에서만 동작)
    ip = get_ipython()
    ip.run_line_magic("load_ext", "autoreload")
    ip.run_line_magic("autoreload", "2")
except Exception:
    pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Audio, display

import audio            # 파일 읽기·16 kHz 모노 변환(수업 코드)
import clip_compare     # 구간 × 소리 × 모델 크기 × VAD 비교
import separate         # 곡 구간 자르기 + 보컬 분리(Demucs)
import visualisation    # 스펙트로그램·Log-Mel 그림

songs = separate.list_songs("en")           # data/songs/en 의 곡 파일(이름순). 번호는 이 목록의 순서(1부터)
print("프로젝트 루트:", Path.cwd().name, "| 영어 곡", len(songs), "개 | 작업 폴더", separate.WORK_DIR)


## 곡 목록 (번호 = `CLIPS`의 `song` 값)

아래 표가 영어 곡 전체 목록입니다. **번호**를 `CLIPS`의 `song`에 적고, **길이(초)**를 보고 `start`가 곡 길이를 넘지 않게 합니다(간주가 긴 곡은 들어 보고 정합니다). 이 표의 출력에는 곡 파일 이름이 나오므로 커밋할 때는 출력을 지운 정리본만 올립니다. 곡을 폴더에 더하거나 빼면 번호가 바뀌니, 구간을 분리한 뒤에는 `data/song_work/clips.csv`의 파일 이름을 기준으로 봅니다.

In [ ]:
# 곡 목록 표: 번호, 파일 이름, 형식, 길이(초), 크기(MB). 길이는 파일 머리글에서 읽어서 빠르다(전체 디코딩 안 함).
import av   # PyAV: faster-whisper가 함께 깐 디코더 패키지. 여기서는 길이만 읽는다.

def song_seconds(path):
    try:
        with av.open(str(path)) as ct:
            return round(ct.duration / 1e6, 1) if ct.duration else None   # duration은 마이크로초 단위
    except Exception:
        return None

song_table = pd.DataFrame({
    "번호": range(1, len(songs) + 1),
    "파일 이름": [p.name for p in songs],
    "형식": [p.suffix.lower().lstrip(".") for p in songs],
    "길이(초)": [song_seconds(p) for p in songs],
    "크기(MB)": [round(p.stat().st_size / 1e6, 1) for p in songs],
})
with pd.option_context("display.max_rows", None, "display.max_colwidth", None, "display.width", 200):
    display(song_table.set_index("번호"))

- 유형별 곡 번호 목록
  - 가사가 또렷한 곡: 12(gospel), 58(bossa nova/folk), 48(rock), 9(ballad), 16(folk)
  - 반주가 두꺼운 곡: 1(brass), 18(synth), 26(live), 15(jazz), 23(hip hop), 53(synth)
  - 합창/코러스/겹치는 목소리: 35(musical/chorus), 51(accapella), 52(duet), 49(chorus), 44(duet)
  - 랩: 55(rap), 3(말+노래), 13(rap)

## ② 구간 정하기

`CLIPS`에 구간을 한 줄씩 적습니다. `song`은 ①의 목록 번호, `start`는 구간 시작(초), `duration`은 길이(초, 10~20 권장), `id`는 구간 이름(`c01`, `c02`, …)입니다. 번호는 폴더의 곡이 바뀌면 달라지므로, 분리하면 `data/song_work/clips.csv`에 **파일 이름**이 함께 기록됩니다.

구간 선택 기준은 `docs/노래_구간_비교_실험_절차.md` 2절입니다(가사가 또렷한 곡, 반주가 두꺼운 곡, 합창이 있는 구간을 섞기).

In [ ]:
# ② 구간 목록. 아래 예시는 주석입니다. 지우고 내 구간으로 채우세요.
CLIPS = [
    # {"id": "c01", "song": 6, "start": 60, "duration": 20},
    # {"id": "c02", "song": 12, "start": 45, "duration": 15},
    {"id": "c01", "song": 9,  "start": 61,  "duration": 20},   # 또렷
    {"id": "c02", "song": 58, "start": 96,  "duration": 20},   # 또렷
    {"id": "c03", "song": 48, "start": 102, "duration": 20},   # 또렷
    {"id": "c04", "song": 26, "start": 60, "duration": 20},   # 반주 두꺼움
    {"id": "c05", "song": 18, "start": 20,  "duration": 20},   # 반주 두꺼움
    {"id": "c06", "song": 1,  "start": 31,  "duration": 20},   # 반주 두꺼움
    {"id": "c07", "song": 51, "start": 71,  "duration": 20},   # 합창 (아카펠라, 대조군)
    {"id": "c08", "song": 35, "start": 60,  "duration": 20},   # 합창
]
OVERWRITE = False       # True면 이미 분리한 구간도 다시 분리합니다(start를 고쳤을 때)

## ③ (이전 방식) 자르기 + 분리를 한 번에 — c01~c08 첫 실행 기록

아래 셀은 **자르기와 분리를 한 번에** 하던 첫 방식입니다. 출력에 분리 시간 기록이 남아 있어 그대로 둡니다. **다시 실행하지 않습니다.** 앞으로는 아래 ③a(자르기) → ④(듣기) → ③b(분리) 순서로 합니다. 구간을 듣고 확인한 뒤에 분리하면 어긋난 구간의 분리 시간(구간당 1분 안팎)을 아낄 수 있습니다.

In [ ]:
# # ③ 구간마다 분리: separate.process_clip → data/song_work/<id>_mix.wav, <id>_vocals.wav, clips.csv
# if not CLIPS:
#     print("CLIPS가 비어 있습니다. ②에서 구간을 적고 다시 실행하세요.")
# for c in CLIPS:
#     mix = Path(separate.WORK_DIR) / f"{c['id']}_mix.wav"
#     if mix.exists() and not OVERWRITE:
#         print(f"{c['id']}: 이미 있음 → 건너뜀 (다시 하려면 OVERWRITE = True)")
#         continue
#     r = separate.process_clip(songs[c["song"] - 1], c["start"], c["duration"], c["id"], overwrite=OVERWRITE)
#     print(f"{c['id']}: 분리 {r['seconds']:.1f}초 ({r['seconds'] / c['duration']:.2f}배)")

## ③a 구간 자르기 (몇 초)

`CLIPS`의 구간을 **자르기만** 합니다(분리 안 함). 곡 번호·`start`·`duration`이 이미 잘라 둔 것과 같은 구간은 건너뛰고, **바뀐 구간만 다시 자릅니다**(그 구간의 옛 보컬 파일은 지웁니다). 그래서 `start`를 고친 뒤 이 셀만 다시 실행하면 되고 `OVERWRITE`는 켜지 않아도 됩니다. 일부 구간만 하려면 `ONLY = ["c05", "c06"]`처럼 적습니다.
다 자른 뒤에는 ④에서 `<id>_mix.wav`를 들어 **가사 부분이 맞는지 확인**합니다.

In [ ]:
# ③a 자르기만: separate.cut_clip → data/song_work/<id>_mix.wav, clips.csv. 같은 구간은 건너뛰고 바뀐 구간만 다시 자른다.
ONLY = None      # None이면 CLIPS 전부. 일부만: 예) ["c05", "c06"]
targets = [c for c in CLIPS if ONLY is None or c["id"] in ONLY]
if not targets:
    print("대상 구간이 없습니다. ②의 CLIPS와 ONLY를 확인하세요.")
for c in targets:
    song = songs[c["song"] - 1]
    if separate.is_cut_current(song, c["start"], c["duration"], c["id"]):
        print(f"{c['id']}: 이미 같은 구간을 잘라 둠 → 건너뜀")
        continue
    r = separate.cut_clip(song, c["start"], c["duration"], c["id"], overwrite=True)   # 바뀐 구간이면 다시 자르고 옛 보컬은 지움
    print(f"{c['id']}: 곡 {c['song']}번 {c['start']}초부터 {r['seconds']:.0f}초 잘랐습니다 → ④에서 {c['id']}_mix.wav를 들어 보세요")

## ③b 분리 (구간당 1~2분)

④에서 **들어 보고 가사 부분이 맞다고 확인한 구간만** 분리합니다. 이미 보컬 파일이 있는 구간은 건너뜁니다. 어긋난 구간은 `start`를 고쳐 ③a를 다시 실행한 뒤(그 구간만 다시 잘림) 이 셀을 실행합니다. 일부만 분리하려면 ③a의 `ONLY`를 바꿉니다. 셀이 끝날 때까지 기다립니다.

**분리는 별도 프로세스(`scripts/separate_clip.py --separate-only`)로 실행합니다.** 분리에 쓰는 `torch`가 이 노트북 커널에 올라가면, 같은 커널에서 ⑥(Whisper 인식)을 실행할 때 OpenMP 충돌(`OMP: Error #15`)로 커널이 죽기 때문입니다(`docs/개발환경_이슈노트.md` #15). 이렇게 하면 ③b와 ⑥을 어떤 순서로 실행해도 됩니다.

In [ ]:
# ③b 분리: 구간마다 터미널 스크립트(scripts/separate_clip.py --separate-only)를 별도 프로세스로 실행한다.
# 왜 subprocess인가: torch(Demucs)를 이 커널에 불러오면 ⑥의 Whisper(CTranslate2)와 OpenMP가 충돌해 커널이 죽는다(이슈노트 #15).
# 별도 프로세스는 끝나면 사라지므로 이 커널에는 torch가 올라가지 않는다.
import subprocess

targets = [c for c in CLIPS if ONLY is None or c["id"] in ONLY]
for c in targets:
    mix_p, voc_p = separate.clip_paths(c["id"])
    if not mix_p.exists():
        print(f"{c['id']}: 아직 자르지 않았습니다 → ③a를 먼저 실행하세요")
        continue
    if voc_p.exists():
        print(f"{c['id']}: 이미 분리함 → 건너뜀")
        continue
    print(f"{c['id']}: 분리 중(1~2분)…")
    r = subprocess.run([sys.executable, "scripts/separate_clip.py", "--id", c["id"], "--separate-only"],
                       capture_output=True, text=True)       # sys.executable = 이 커널의 파이썬(voice-ai). 출력은 아래에서 골라 보여 준다
    if r.returncode != 0:
        print(f"{c['id']}: 실패(종료 코드 {r.returncode})")
        print("\n".join(r.stderr.strip().splitlines()[-5:]))   # 오류 마지막 5줄
        continue
    print("   " + " ".join(l for l in r.stdout.splitlines() if l.startswith("분리")))

## ④ 들어 보기

`LISTEN_ID`를 바꿔 구간을 듣습니다. **③a 직후에는 원곡(mix)만** 있으니 먼저 mix로 구간이 가사 부분인지 확인하고, 맞으면 ③b로 분리한 뒤 보컬(vocals)도 들어 분리 품질(좋음/보통/나쁨)을 봅니다. 어긋나면 `start`를 고치고 ③a를 다시 실행합니다. (이 셀의 출력에는 곡 소리가 들어가므로 커밋 전에 출력을 지웁니다.)

In [ ]:
# ④ 듣기: 구간 원본(mix)과 분리한 보컬(vocals)
LISTEN_ID = "c03"
for name in ("mix", "vocals"):
    p = Path(separate.WORK_DIR) / f"{LISTEN_ID}_{name}.wav"
    if p.exists():
        print(name)
        display(Audio(str(p)))
    else:
        print(f"{p.name} 이 없습니다. ③a(mix) 또는 ③b(vocals)를 먼저 실행하세요.")

## ⑤ 정답 가사 준비 (직접 받아 적기)

아래 셀은 정답 파일 `data/song_work/truth/<id>.txt`의 **빈 파일을 만들고 현재 상태(단어 수)만** 보여 줍니다. 파일은 **VS Code나 텍스트 편집기에서 직접 열어** 구간에서 불린 가사의 단어를 받아 적습니다. **정답은 분리 전 원곡(mix)을 들으며 적습니다**(분리 결과에 정답이 끌려가지 않게 하려는 것)(규칙은 절차 문서 4절: 추임새 제외, 반복은 횟수대로). 가사는 이 화면에 나오지 않습니다. 다 적고 이 셀을 다시 실행해 단어 수와 "준비됨"을 확인합니다.

In [ ]:
# ⑤ 정답 파일 만들기와 상태 표(가사는 찍지 않고 단어 수만)
ids = [c["id"] for c in CLIPS]
truth_dir = Path(separate.WORK_DIR) / "truth"
truth_dir.mkdir(parents=True, exist_ok=True)
rows = []
for cid in ids:
    t = clip_compare.truth_path(cid)
    if not t.exists():
        t.touch()                                           # 빈 파일을 만들어 둔다(편집기로 열 수 있게)
    n_words = len(clip_compare.load_truth(cid).split())
    rows.append({"구간": cid, "정답 파일": str(t), "정답 단어 수": n_words})
ready, missing = clip_compare.available_clips()
for r in rows:
    r["준비됨"] = r["구간"] in ready
    r["빠진 것"] = "/".join(missing.get(r["구간"], [])) or "-"
with pd.option_context("display.max_colwidth", None, "display.width", 200):
    display(pd.DataFrame(rows))

## ⑥ 비교 실행

준비된 구간을 **원곡 / 보컬 × 모델 크기 × VAD**로 인식하고 정답과 비교합니다. 로컬이라 과금은 없습니다. 구간 5개 × 2 × 크기 3 × VAD 2 = 60번 인식에 10~20분 [추정]이 걸릴 수 있으니, 처음에는 `SIZES = ["tiny"]`, `VADS = (False,)`로 짧게 확인합니다. `small`은 처음 쓸 때 내려받습니다.

화면 표의 읽는 법은 절차 문서 6절입니다. 인식된 문장(가사)은 `results/song_compare/texts.csv`에만 저장되고 화면에는 나오지 않습니다.

무엇을 어떻게 비교하는지(소리 × 모델 크기 × VAD, 정답, 점수)는 맨 위 「이 노트북은 무엇을 하는가」에 정리해 두었습니다. 같은 입력이어도 결과가 실행마다 달라질 수 있으므로(온도 폴백) `temperature` 옵션을 추가한 뒤 그 설정을 함께 적어 보고합니다.


In [ ]:
# ⑥ 비교 실행: clip_compare.compare → 점수표(metrics, 가사 없음)와 인식 문장(texts, 가사)
SIZES = ["tiny", "base", "small"]          # 예: ["tiny", "base", "small"] / ["tiny"]  
VADS = (False,)             # 예: (False, True)  VAD = 말소리 없는 구간을 걸러내는 처리

ready, missing = clip_compare.available_clips()
targets = [i for i in ready if not CLIPS or i in {c["id"] for c in CLIPS}]
if missing:
    print("준비가 덜 된 구간(건너뜀):", {k: "/".join(v) for k, v in missing.items()})
if not targets:
    print("비교할 구간이 없습니다. ③ 분리와 ⑤ 정답을 먼저 준비하세요.")
else:
    metrics, texts = clip_compare.compare(targets, SIZES, VADS)
    out = Path("results/song_compare")
    out.mkdir(parents=True, exist_ok=True)
    metrics.to_csv(out / "metrics.csv", index=False)
    texts.to_csv(out / "texts.csv", index=False)       # 가사: 화면에 찍지 않고 파일로만 저장
    print(f"구간 {len(targets)}개 × 소리 2 × 크기 {len(SIZES)} × VAD {len(VADS)} = {len(metrics)}번 인식 완료")
    print("[1] 조건별 요약 (micro = 오류 합 ÷ 정답 단어 합, macro = 구간별 평균)")
    display(clip_compare.summarize(metrics).round(3))
    print("[2] 구간 × 조건 WER")
    display(clip_compare.pivot_wer(metrics).round(3))

## ⑦ 스펙트로그램: 원곡 vs 분리한 보컬

같은 설정·같은 색 범위로 파형, 스펙트로그램, Log-Mel을 나란히 그립니다(`visualisation.plot_sound_pair`, 4강 코드). 반주가 있는 구간과 없는 구간의 에너지 분포를 비교하고, 한 줄 해석을 절차 문서 8.3절에 적습니다. 그림 제목에는 곡 이름을 넣지 않습니다.

In [ ]:
# ⑦ 원곡 vs 보컬 그림 (16 kHz 모노로 바꿔 비교; 최대 진폭을 0.9로 맞춰 밝기를 비교하기 쉽게)
SPEC_ID = "c01"
mix_p = Path(separate.WORK_DIR) / f"{SPEC_ID}_mix.wav"
voc_p = Path(separate.WORK_DIR) / f"{SPEC_ID}_vocals.wav"
if mix_p.exists() and voc_p.exists():
    def norm09(x):
        return (0.9 * x / (np.abs(x).max() + 1e-9)).astype(np.float32)
    mix16 = norm09(audio.load_audio(mix_p)[0])
    voc16 = norm09(audio.load_audio(voc_p)[0])
    fig = visualisation.plot_sound_pair([("원곡 구간(mix)", mix16), ("분리한 보컬(vocals)", voc16)],
                                        f"구간 {SPEC_ID}: 원곡 vs 분리한 보컬")
    plt.show()
else:
    print("먼저 ③ 분리를 실행하세요.")

## 기록

숫자와 관찰은 `docs/노래_구간_비교_실험_절차.md` 8절 표에 적고(가사·곡 이름 없이), 판단은 검토 문서 13.5에 옮깁니다. 이 노트북을 커밋할 때는 **출력이 없는 정리본**만 올립니다(`scripts/scan_outputs.py`로 확인).